# 🎂 Age & Gender Estimator: training notebook
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/age-estimator/training/notebook.ipynb)

This notebook trains the model behind the [Space](https://huggingface.co/spaces/shalev396/age-estimator):
an ImageNet EfficientNet-B2 fine-tuned on **UTKFace** with two heads, one over **90 age bins**
(`0-1`, every year `2`…`89`, `90+`; the headline age is the expected value) and one for **gender**.
At inference our face detector (trained in the face-recognition project) finds the faces first, and Grad-CAM shows what drove each estimate.

Every step calls code from `src/` and the deployed `../model/model.py`, so training, the Space and the
Inference Endpoint share one implementation. Set `SMOKE_TEST=1` for a 2-minute run on a tiny slice.

**Contents:** 1 Setup · 2 Config · 3 Data · 4 Load model · 5 Training · 6 Evaluation · 7 Inference · 8 Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "age-estimator"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## 2. Config
Every hyperparameter lives in `src/config.py`. `SMOKE_TEST` (env var or the line below) shrinks the run to
~100 faces and 1 epoch and sends every output to `outputs/smoke/`, so the deployed `../model` is never touched.
`PUSH_TO_HUB` uploads the exported folder at the end (needs `hf auth login` or an `HF_TOKEN` secret/env var).
`OVERWRITE_IF_WORSE` lets a full run replace `../model` even when its validation age MAE is not lower.

In [ ]:
import dataclasses
import time

import pandas as pd
import torch
from IPython.display import display

from src import data_setup, engine, export, model_builder, utils
from src.config import Config
import model as M  # ../model/model.py (put on sys.path by src/__init__.py)

SMOKE_TEST = os.getenv("SMOKE_TEST", "0") == "1"   # True -> 2-minute sanity run
PUSH_TO_HUB = False
OVERWRITE_IF_WORSE = False   # True -> export to ../model even if this run is worse

cfg = Config(smoke=SMOKE_TEST)
device = torch.device(device)
print(utils.lib_versions("torch", "torchvision", "huggingface_hub", "numpy"), "| device:", device.type)
pd.Series(dataclasses.asdict(cfg)).to_frame("value")

## 3. Data
UTKFace from the Hugging Face Hub (`nu-delta/utkface`, 3 parquet shards, ~1 GB), cached in `data/utkface/`.
It holds 23,705 aligned face crops (200×200) with age 1–116 and gender. The split is a 90/10 random
train/validation split with seed 42 (the same split the deployed weights were selected on).
UTKFace is licensed for non-commercial research only.

In [ ]:
files = data_setup.download_data(cfg)
table = data_setup.load_table(cfg, files)
train_ds, val_ds = data_setup.build_datasets(cfg, table)
age_reps = data_setup.compute_age_representatives(train_ds.ages)
print([f.name for f in files])
data_setup.summary(train_ds, val_ds)

In [ ]:
from IPython.display import Image as ShowImage
eda_dir = cfg.run_dir / "eda"
ShowImage(filename=str(export.plot_age_distribution(train_ds.ages, eda_dir / "age_distribution.png",
                                                    "Training faces by age")))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
for ax, i in zip(axes, range(0, len(train_ds), max(1, len(train_ds) // 6))):
    ax.imshow(train_ds.image(i)); ax.axis("off")
    ax.set_title(f"{train_ds.ages[i]} y, {M.DEFAULT_GENDER_LABELS[train_ds.genders[i]]}", fontsize=9)
plt.show()
train_loader, val_loader = data_setup.create_dataloaders(cfg, train_ds, val_ds, device)
print(f"{len(train_loader)} train batches / {len(val_loader)} validation batches of {cfg.batch_size}")

## 4. Load model
`model.AgeGenderNet` from `../model/model.py`: EfficientNet-B2 trunk (ImageNet weights) + `Dropout(0.3) → Linear(1408, 90)`
age head + `Dropout(0.3) → Linear(1408, 2)` gender head. The per-bin representative ages computed from the
training faces are stored in the model config (they turn the age softmax into a single number).

In [ ]:
net = model_builder.build_model(cfg, age_reps).to(device)
print(f"parameters: {utils.count_params(net):,} ({utils.human_params(utils.count_params(net))}) | "
      f"age bins: {len(net.age_labels)} ({net.age_labels[0]}, {net.age_labels[1]} … {net.age_labels[-1]})")

## 5. Training
Loss = cross-entropy over the age bins + cross-entropy over gender. Adam, full fine-tune, mixed precision on
CUDA. After every epoch the validation age MAE is measured and the best epoch's weights are kept.

In [ ]:
start = time.perf_counter()
history = engine.fit(net, train_loader, val_loader, cfg, device, age_reps)
train_time_s = time.perf_counter() - start
print(f"training time: {train_time_s:.0f} s")
ShowImage(filename=str(export.plot_history(history, cfg.run_dir / "training_curves.png")))

## 6. Evaluation
Validation metrics of the best epoch: age **MAE** (years, expected-age decoding), RMSE, share of faces within
±5 years, gender accuracy. Compared with argmax-bin decoding of the same network, a no-learning baseline
(median training age + majority gender) and the model currently deployed in `../model`.

In [ ]:
ev = engine.evaluate(net, val_loader, device, age_reps, cfg.amp)
scores = engine.scores(ev["true_age"], ev["age_pred"], ev["gender_true"], ev["gender_pred"])
base = model_builder.baseline_predictions(train_ds.ages, train_ds.genders, len(val_ds))
run_name = "this run" + (" (smoke)" if cfg.smoke else "")
comparison = {
    f"EfficientNet-B2, expected age ({run_name})": scores,
    f"EfficientNet-B2, argmax bin ({run_name})": engine.scores(ev["true_age"], ev["age_pred_argmax"],
                                                              ev["gender_true"], ev["gender_pred"]),
    "baseline: median age + majority gender": engine.scores(ev["true_age"], base["age_pred"],
                                                            ev["gender_true"], base["gender_pred"]),
}
deployed = utils.load_json(utils.MODEL_DIR / "metrics.json") if (utils.MODEL_DIR / "metrics.json").is_file() else None
table_rows = dict(comparison)
if deployed:
    table_rows["deployed model in ../model"] = deployed["metrics"]
pd.DataFrame(table_rows).T.style.format("{:.4f}")

In [ ]:
mae_groups = engine.mae_by_age_group(ev["true_age"], ev["age_pred"])
display(pd.DataFrame(mae_groups).T)
plots = export.write_assets(cfg.run_dir / "assets", {"comparison": comparison}, ev, None, mae_groups)
for p in plots:
    display(ShowImage(filename=str(p)))

## 7. Inference
Save the trained network as a loadable folder (weights + config + `model.py` + face detector) and run
`model.load(dir, device).predict(image)` on `../space/examples/`, exactly what the Space and the endpoint run.
Then the Grad-CAM views the Space shows (red = regions that decided the estimate).

In [ ]:
run_model_dir = export.save_model_files(net, cfg.run_dir / "trained_model")
predictor = M.load(run_model_dir, device.type)
examples = sorted((utils.SPACE_DIR / "examples").glob("*.jpg"))
results = {p.name: predictor.predict(p) for p in examples}
for name, r in results.items():
    f = r["faces"][0]
    print(f"{name}: {r['n_faces']} face(s) | age {f['age']} ({f['age_range']}) | gender {f['gender']}")

In [ ]:
from PIL import Image
fig, axes = plt.subplots(len(examples), 3, figsize=(9, 3.1 * len(examples)))
for row, p in zip(axes, examples):
    img = Image.open(p).convert("RGB")
    cams = predictor.gradcam(img, results[p.name])
    for ax, pic, title in zip(row, (M.annotate(img, results[p.name]), cams["age"], cams["gender"]),
                              ("prediction", "age evidence", "gender evidence")):
        ax.imshow(pic); ax.axis("off"); ax.set_title(f"{p.name}: {title}", fontsize=8)
plt.tight_layout(); plt.show()

## 8. Export
Write weights, `config.json`, `metrics.json` and the card plots to `cfg.model_dir` (`../model` for a full run,
`outputs/smoke/model` for a smoke run) and refresh the metrics in its `README.md`. `export.export` checks
`../model/metrics.json` first: it replaces the deployed model only when this run's validation age MAE is lower
(or `OVERWRITE_IF_WORSE = True`); otherwise it writes nothing and says so.

In [ ]:
metrics = export.build_metrics(
    scores,
    data={"n_train": len(train_ds), "n_val": len(val_ds), "n_age_bins": len(net.age_labels), "n_gender_classes": 2},
    params=utils.count_params(net),
    train_time_s=train_time_s,
    device=device.type,
    smoke=cfg.smoke,
    source=f"retrained {utils.today()} with training/ ({device.type})",
    comparison=comparison,
    extra={"best_epoch": history["best_epoch"], "epochs": cfg.epochs, "age_mae_by_group": mae_groups},
)
exported = export.export(net, cfg.model_dir, metrics, evaluation=ev, history=history, mae_groups=mae_groups,
                         overwrite_if_worse=OVERWRITE_IF_WORSE)
sorted(str(p.relative_to(cfg.model_dir)) for p in cfg.model_dir.rglob("*") if p.is_file() and "__pycache__" not in p.parts)

In [ ]:
# Publish: uploads cfg.model_dir to huggingface.co/shalev396/age-estimator (full runs only).
# Token: `hf auth login` locally, or an HF_TOKEN secret / env var (e.g. Colab secrets).
if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))
else:
    print("not uploading (PUSH_TO_HUB is False or this is a smoke run)")